In [ ]:
import os

import cmcrameri.cm as cmc
import h5py
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from scipy.ndimage import gaussian_filter1d

from ptycho_fm.metrics import (
    analyze_independent_erf_edges,
    sample_parallel_line_profiles,
)

%matplotlib inline

## Scan 267 (Supplementary Video 1)

In [ ]:
scan267_dir = "/mnt/localdisk/scratch/aileenluo/recon_20260225"

# DM
with h5py.File(os.path.join(scan267_dir, "scan_267_dm/scan_267_epoch20_para.hdf5"), "r") as f:
    dm = f["object"][0, 154:-154, 154:-154]
amp_dm = np.abs(dm)
ph_dm = np.angle(dm)

# RAAR
with h5py.File(os.path.join(scan267_dir, "scan_267_raar/scan_267_epoch15_para.hdf5"), "r") as f:
    raar = f["object"][0, 154:-154, 154:-154]
amp_raar = np.abs(raar)
ph_raar = np.angle(raar)

# ePIE
with h5py.File(os.path.join(scan267_dir, "scan_267_epie/scan_267_epoch3000_para.hdf5"), "r") as f:
    epie = f["object"][0, 154:-154, 154:-154]
amp_epie = np.abs(epie)
ph_epie = np.angle(epie)

# rPIE
with h5py.File(os.path.join(scan267_dir, "scan_267_rpie/scan_267_epoch35_para.hdf5"), "r") as f:
    rpie = f["object"][0, 154:-154, 154:-154]
amp_rpie = np.abs(rpie)
ph_rpie = np.angle(rpie)

# LSQML
with h5py.File(os.path.join(scan267_dir, "scan_267_lsqml/scan_267_epoch900_para.hdf5"), "r") as f:
    lsqml = f["object"][0, 154:-154, 154:-154]
amp_lsqml = np.abs(lsqml)
ph_lsqml = np.angle(lsqml)

inference_dir = "/mnt/localdisk/scratch/aileenluo/results"

# Flyscan inference 43M fine-tuned
amp_43M_finetune = np.load(os.path.join(inference_dir, "run164/pred_amp_scan_267_datanorm.npy"))
ph_43M_finetune = np.load(os.path.join(inference_dir, "run164/pred_ph_scan_267_datanorm.npy"))

# Flyscan inference 200M fine-tuned
amp_200M_finetune = np.load(os.path.join(inference_dir, "run167/pred_amp_scan_267_datanorm.npy"))
ph_200M_finetune = np.load(os.path.join(inference_dir, "run167/pred_ph_scan_267_datanorm.npy"))

## Edge-resolution analysis

Independent error-function fits report absolute, model-dependent edge widths.
The LSQML reference profile is used only to locate the edge and define a common
fitting window along the selected line and parallel-cut averaging strip.


### Prepare phase and amplitude arrays

All learned inference arrays—every name containing `43M` or `200M`—are flipped vertically with `np.flipud` before the edge analysis. Iterative reconstructions retain their loaded orientation.


In [ ]:
MODEL_NAME_MARKERS = ('43M', '200M')


def orient_for_resolution(images):
    """Apply the established orientation convention to model inferences."""
    return {
        name: np.flipud(image)
        if any(marker in name for marker in MODEL_NAME_MARKERS)
        else image
        for name, image in images.items()
    }


raw_phase_images = {
    'ph_lsqml': ph_lsqml,
    'ph_dm': ph_dm,
    'ph_raar': ph_raar,
    'ph_epie': ph_epie,
    'ph_rpie': ph_rpie,
    'ph_43M_finetune': ph_43M_finetune,
    'ph_200M_finetune': ph_200M_finetune,
}
raw_amplitude_images = {
    'amp_lsqml': amp_lsqml,
    'amp_dm': amp_dm,
    'amp_raar': amp_raar,
    'amp_epie': amp_epie,
    'amp_rpie': amp_rpie,
    'amp_43M_finetune': amp_43M_finetune,
    'amp_200M_finetune': amp_200M_finetune,
}

phase_images = orient_for_resolution(raw_phase_images)
amplitude_images = orient_for_resolution(raw_amplitude_images)

flipped_model_count = 0
for oriented_images, raw_images in (
    (phase_images, raw_phase_images),
    (amplitude_images, raw_amplitude_images),
):
    model_names = [
        name
        for name in oriented_images
        if any(marker in name for marker in MODEL_NAME_MARKERS)
    ]
    flipped_model_count += len(model_names)
    assert all(
        np.array_equal(
            oriented_images[name], np.flipud(raw_images[name]), equal_nan=True
        )
        for name in model_names
    )

print(
    'Prepared phase and amplitude images; '
    f'flipped {flipped_model_count} model inference arrays.'
)


### User settings: choose the line cut

Edit `line_start_rc` and `line_end_rc` below as `(row, column)` coordinates. The same physical edge should be present in every reconstruction; the preview cell lets you verify the selected cut on `ph_lsqml` before running the fits. The strip width and fit controls can also be adjusted here.


In [ ]:
# USER SETTINGS: endpoints are (row, column) in the cropped arrays.
line_start_rc = (1560, 265) #(1220, 510) #(2390, 2300) # dx: 80, dy: 120
line_end_rc = (1640, 385) #(1300, 630) #(2470, 2420) 

phase_reference_name = 'ph_lsqml'
pixel_size_nm = 6.685

# Shared sampling and fit settings.
n_samples = 600
strip_width_px = 10
n_parallel_cuts = 11
reference_edge_nm = None  # None locates the strongest gradient in the reference profile.
fit_half_width_nm = 150
shift_tolerance_nm = 50
sigma_min_px = 0.25
minimum_r_squared = 0.8
minimum_edge_cnr = 3.0
maximum_relative_sigma_uncertainty = 0.5
erf_n_steps = 600


def validate_line_cut(images, start_rc, end_rc, width_px):
    shapes = {name: np.asarray(image).shape for name, image in images.items()}
    if len(set(shapes.values())) != 1:
        raise ValueError(f'All images must share one shape; received {shapes}')
    if start_rc == end_rc:
        raise ValueError('line_start_rc and line_end_rc must differ')
    rows, cols = next(iter(shapes.values()))
    margin = width_px / 2
    for label, (row, col) in (('start', start_rc), ('end', end_rc)):
        if not (margin <= row <= rows - 1 - margin):
            raise ValueError(f'{label} row {row} is outside the safe image bounds')
        if not (margin <= col <= cols - 1 - margin):
            raise ValueError(f'{label} column {col} is outside the safe image bounds')
    return rows, cols


image_shape = validate_line_cut(
    phase_images, line_start_rc, line_end_rc, strip_width_px
)
validate_line_cut(
    amplitude_images, line_start_rc, line_end_rc, strip_width_px
)
print(
    f'Using line {line_start_rc} → {line_end_rc} on images with shape '
    f'{image_shape}.'
)

In [ ]:
# Preview the designated line cut on the phase reference.
y0, x0 = line_start_rc
y1, x1 = line_end_rc
line_preview_fig, line_preview_ax = plt.subplots(figsize=(8, 7))
line_preview_image = line_preview_ax.imshow(
    phase_images[phase_reference_name], interpolation='nearest', cmap=cmc.grayC
)
line_preview_ax.plot(
    [x0, x1], [y0, y1], color='#8F2D3C', linewidth=1.8,
    label='designated line cut',
)
line_preview_ax.scatter(
    [x0, x1], [y0, y1], color='#8F2D3C', s=35, zorder=5
)
divider = make_axes_locatable(line_preview_ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
line_preview_fig.colorbar(line_preview_image, cax=cax)
line_preview_ax.set_title(f'Line-cut preview on {phase_reference_name}')
line_preview_ax.legend(loc='upper left')
line_preview_fig.tight_layout()
plt.show()

### Locate the reference edge and select the erf fitting window

Sample the LSQML reference along the chosen strip. When `reference_edge_nm` is
`None`, smooth this profile over approximately one native pixel and locate its
strongest gradient. Center the shared fitting window on that position, clipping
the window to the sampled line. This selects the same window as before without
performing a reference-template broadening fit.


In [ ]:
reference_samples = sample_parallel_line_profiles(
    phase_images[phase_reference_name],
    line_start_rc,
    line_end_rc,
    pixel_size_nm=pixel_size_nm,
    n_samples=n_samples,
    strip_width_px=strip_width_px,
    n_parallel_cuts=n_parallel_cuts,
)
reference_distance_nm = reference_samples['distance_nm']
if reference_edge_nm is None:
    samples_per_native_px = (n_samples - 1) / reference_samples['line_length_px']
    smoothed_reference_profile = gaussian_filter1d(
        reference_samples['mean_profile'],
        sigma=max(samples_per_native_px, 1.0),
    )
    reference_gradient = np.gradient(smoothed_reference_profile, reference_distance_nm)
    phase_reference_edge_nm = float(
        reference_distance_nm[np.argmax(np.abs(reference_gradient))]
    )
else:
    phase_reference_edge_nm = float(reference_edge_nm)

if fit_half_width_nm <= 0:
    raise ValueError('fit_half_width_nm must be positive')
phase_fit_interval_nm = (
    max(float(reference_distance_nm[0]), phase_reference_edge_nm - fit_half_width_nm),
    min(float(reference_distance_nm[-1]), phase_reference_edge_nm + fit_half_width_nm),
)
if phase_fit_interval_nm[0] >= phase_fit_interval_nm[1]:
    raise ValueError('The fit window must overlap the sampled line')
print(
    f'Reference edge: {phase_reference_edge_nm:.2f} nm; '
    f'erf fitting window: {phase_fit_interval_nm[0]:.2f}–{phase_fit_interval_nm[1]:.2f} nm'
)


### Summary of independent phase error-function fits

In [ ]:
method_names = [
    'ph_43M_finetune',
    'ph_dm',
    'ph_raar',
    'ph_epie',
    'ph_rpie',
    'ph_lsqml',
]
method_labels = {
    'ph_43M_finetune': 'PtychoFM',
    'ph_dm': 'DM',
    'ph_raar': 'RAAR',
    'ph_epie': 'ePIE',
    'ph_rpie': 'rPIE',
    'ph_lsqml': 'LSQML',
}
line_color = '#0072B2'

# Fit the selected methods independently in the shared reference-edge window.
phase_erf_analysis = analyze_independent_erf_edges(
    {name: phase_images[name] for name in method_names},
    phase_reference_name,
    line_start_rc,
    line_end_rc,
    pixel_size_nm=pixel_size_nm,
    fit_interval_nm=phase_fit_interval_nm,
    reference_edge_nm=phase_reference_edge_nm,
    shift_tolerance_nm=shift_tolerance_nm,
    n_steps=erf_n_steps,
    strip_width_px=strip_width_px,
    n_parallel_cuts=n_parallel_cuts,
    sigma_min_px=sigma_min_px,
    minimum_r_squared=minimum_r_squared,
    minimum_edge_cnr=minimum_edge_cnr,
    maximum_relative_sigma_uncertainty=maximum_relative_sigma_uncertainty,
)

# Geometry of the 300 nm line segment used by the fits.
geometry = phase_erf_analysis['sampled']['ph_43M_finetune']
row_start, row_end = geometry['rows'][[0, -1]]
col_start, col_end = geometry['cols'][[0, -1]]
normal_row = geometry['normal_row']
normal_col = geometry['normal_col']
half_strip_width_px = strip_width_px / 2
strip_rows = np.array([
    row_start + half_strip_width_px * normal_row,
    row_end + half_strip_width_px * normal_row,
    row_end - half_strip_width_px * normal_row,
    row_start - half_strip_width_px * normal_row,
])
strip_cols = np.array([
    col_start + half_strip_width_px * normal_col,
    col_end + half_strip_width_px * normal_col,
    col_end - half_strip_width_px * normal_col,
    col_start - half_strip_width_px * normal_col,
])

# Define an exact 500 × 500 pixel square centered on the fit segment.
image = np.asarray(phase_images['ph_43M_finetune'])
square_size_px = 500
square_center_row = 0.5 * (row_start + row_end)
square_center_col = 0.5 * (col_start + col_end)
square_row_lo = int(np.clip(
    round(square_center_row - square_size_px / 2),
    0, image.shape[0] - square_size_px,
))
square_col_lo = int(np.clip(
    round(square_center_col - square_size_px / 2),
    0, image.shape[1] - square_size_px,
))
square_row_hi = square_row_lo + square_size_px
square_col_hi = square_col_lo + square_size_px
zoom_square = image[
    square_row_lo:square_row_hi, square_col_lo:square_col_hi
]
phase_vmin = np.mean(image) - (3 * np.std(image))
phase_vmax = np.mean(image) + (3 * np.std(image))

# Figure 1: full phase array and the corresponding 500 × 500 pixel slice.
phase_array_fig, (full_ax, square_ax) = plt.subplots(
    ncols=2, figsize=(7, 3), dpi=300,
)
full_artist = full_ax.imshow(
    image, cmap=cmc.grayC, interpolation='none',
    vmin=phase_vmin, vmax=phase_vmax,
)
full_ax.add_patch(Rectangle(
    (square_col_lo - 0.5, square_row_lo - 0.5),
    square_size_px, square_size_px, fill=False,
    edgecolor=line_color, linewidth=1.6,
))
full_ax.fill(
    strip_cols, strip_rows, color=line_color,
    alpha=0.5, linewidth=0,
)
full_ax.plot(
    [col_start, col_end], [row_start, row_end],
    color=line_color, linewidth=2.0,
)
divider = make_axes_locatable(full_ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
phase_array_fig.colorbar(full_artist, cax=cax, orientation='vertical')
full_ax.set_title('Full phase array')
full_ax.set_xticks([])
full_ax.set_yticks([])
full_scale_bar = AnchoredSizeBar(
    full_ax.transData, 2000 / pixel_size_nm, '2 µm',
    loc='lower right', pad=0.2, color='#E8E8E8', frameon=False,
    size_vertical=5
)
full_ax.add_artist(full_scale_bar)

square_artist = square_ax.imshow(
    zoom_square, cmap=cmc.grayC, interpolation='none',
    vmin=phase_vmin, vmax=phase_vmax,
    extent=(
        square_col_lo - 0.5, square_col_hi - 0.5,
        square_row_hi - 0.5, square_row_lo - 0.5,
    )
)
square_ax.fill(
    strip_cols, strip_rows, color=line_color,
    alpha=0.5, linewidth=0,
)
square_ax.plot(
    [col_start, col_end], [row_start, row_end],
    color=line_color, linewidth=2.0,
)
square_ax.scatter(
    [col_start, col_end], [row_start, row_end],
    s=22, color=line_color, edgecolor='white',
    linewidth=0.6, zorder=3,
)
square_ax.set_title('500 × 500 pixel slice')
square_ax.set_xticks([])
square_ax.set_yticks([])
square_scale_bar = AnchoredSizeBar(
    square_ax.transData, 500 / pixel_size_nm, '500 nm',
    loc='lower right', pad=0.2, color='#E8E8E8', frameon=False,
    size_vertical=1
)
square_ax.add_artist(square_scale_bar)
divider = make_axes_locatable(square_ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
phase_array_fig.colorbar(square_artist, cax=cax, orientation='vertical')
plt.show()
#phase_array_fig.savefig("resolution_a.svg", transparent=True, bbox_inches="tight", dpi=300)

# Figure 2: six separate error-function fit panels.
erf_fits_fig, erf_fit_axes = plt.subplots(
    nrows=2, ncols=3, figsize=(7, 4.8), sharex=True, sharey=True,
    constrained_layout=True, dpi=300
)
erf_fit_axes = erf_fit_axes.ravel()
distance_nm = phase_erf_analysis['distance_nm']
fit_min_nm, fit_max_nm = phase_erf_analysis['fit_interval_nm']
for panel_index, (name, fit_ax) in enumerate(zip(
    method_names, erf_fit_axes,
)):
    profile = phase_erf_analysis['profiles'][name]
    sem = phase_erf_analysis['profile_sems'][name]
    result = phase_erf_analysis['results'][name]
    fit_ax.plot(
        distance_nm, profile, color=line_color, linewidth=1.05,
        label='Interpolated profile',
    )
    fit_ax.fill_between(
        distance_nm, profile - sem, profile + sem,
        color=line_color, alpha=0.25, linewidth=0, label='±1 SEM',
    )
    if result['success']:
        fit_ax.plot(
            result['fit_t'], result['fit_curve'], color=line_color,
            linewidth=1.8, linestyle='--', label='Erf fit',
        )
        fit_ax.axvline(
            result['t0'], color=line_color, linewidth=1.1,
            linestyle=':', alpha=0.95, label='Edge location',
        )
        fit_label = (
            f"{method_labels[name]}\n"
            f"σ = {result['sigma']:.1f} ± {result['perr'][2]:.1f} nm, "
            f"R² = {result['r_squared']:.3f}"
        )
    else:
        fit_label = f"{method_labels[name]}\nfit failed"
    fit_ax.set_xlim(fit_min_nm, fit_max_nm)
    fit_ax.set_title(fit_label, fontsize=7.8)
    if panel_index >= 3:
        fit_ax.set_xlabel('Distance (nm)', fontsize=8)
    if panel_index % 3 == 0:
        fit_ax.set_ylabel('Phase (rad)', fontsize=8)
    fit_ax.grid(alpha=0.18, linewidth=0.5)
    for spine in fit_ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
    fit_ax.tick_params(
        axis='both', which='both', direction='in', top=True, right=True,
        labelsize=7,
    )
    if panel_index == 0:
        fit_ax.legend(
            loc='best', fontsize=6.5, frameon=False, handlelength=2.0,
        )
plt.show()
#erf_fits_fig.savefig("resolution_b.svg", transparent=True, bbox_inches="tight", dpi=300)

### 500 × 500 pixel slices and line cuts for all six methods

Panels follow the same row-major order as `resolution_b.svg`: PtychoFM, DM,
RAAR, ePIE, rPIE, LSQML. Every panel uses the same crop coordinates and fitted
line segment, including the parallel-cut averaging strip. Each method uses its
full phase image's mean ± 3 standard deviations for display, matching the
single-method slice plot above.


In [ ]:
slice_comparison_fig, slice_comparison_axes = plt.subplots(
    nrows=2, ncols=3, figsize=(7, 4.8), dpi=300, layout='constrained',
)
slice_comparison_fig.get_layout_engine().set(wspace=0.12)
slice_comparison_axes = slice_comparison_axes.ravel()

for method_name, slice_ax in zip(method_names, slice_comparison_axes):
    method_image = np.asarray(phase_images[method_name])
    method_slice = method_image[
        square_row_lo:square_row_hi, square_col_lo:square_col_hi
    ]
    if method_slice.shape != (square_size_px, square_size_px):
        raise ValueError(f'Unexpected slice shape for {method_name}: {method_slice.shape}')
    method_mean = float(np.mean(method_image))
    method_std = float(np.std(method_image))
    slice_artist = slice_ax.imshow(
        method_slice,
        cmap=cmc.grayC,
        interpolation='none',
        vmin=method_mean - 3 * method_std,
        vmax=method_mean + 3 * method_std,
        extent=(
            square_col_lo - 0.5, square_col_hi - 0.5,
            square_row_hi - 0.5, square_row_lo - 0.5,
        ),
    )
    slice_ax.fill(
        strip_cols, strip_rows, color=line_color, alpha=0.5, linewidth=0,
    )
    slice_ax.plot(
        [col_start, col_end], [row_start, row_end],
        color=line_color, linewidth=2.0,
    )
    slice_ax.scatter(
        [col_start, col_end], [row_start, row_end],
        s=22, color=line_color, edgecolor='white', linewidth=0.6, zorder=3,
    )
    slice_ax.set_title(method_labels[method_name], fontsize=9)
    slice_ax.set_xticks([])
    slice_ax.set_yticks([])
    slice_ax.add_artist(AnchoredSizeBar(
        slice_ax.transData, 500 / pixel_size_nm, '500 nm',
        loc='lower right', pad=0.2, color='#E8E8E8', frameon=False,
        size_vertical=1,
    ))
    slice_divider = make_axes_locatable(slice_ax)
    slice_cax = slice_divider.append_axes('right', size='5%', pad=0.05)
    slice_comparison_fig.colorbar(slice_artist, cax=slice_cax, orientation='vertical')
    slice_cax.tick_params(labelsize=6)

slice_comparison_fig.savefig('resolution_c.svg', transparent=True, bbox_inches='tight', dpi=300)
plt.show()
